
# ALDI Web Scraper

This notebook collects product and price information from ALDI using Selenium.

### How it works

1. Searches for the 20 product terms defined in `SEARCH_TERMS`.
2. Loads additional products through pagination and automatic retries.
3. Extracts product IDs, names, brands, selling prices, quantities, unit-price information, URLs, and timestamps.
4. Combines the collected results into a RAW dataset.
5. Validates collection completeness and saves RAW files.

### Data Storage

- Current RAW: `data/raw/aldi_products_raw.csv`
- Historical RAW: `data/history/aldi/YYYY-MM-DD_raw.csv`


Deduplication, quantity normalization, unit-price calculation, classification, and historical price processing are handled by the central pipeline.

### Collection Policy

- Complete collections update the current RAW and create a historical snapshot.
- Partial collections are archived without replacing the current RAW.
- Existing historical snapshots are never overwritten.
- TEST_MODE allows testing selected searches without saving files.


In [ ]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    TimeoutException
)

HOME_URL = "https://www.aldi.es/"
WAIT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche", "yogur", "garbanzos", "detergente"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms:", TERMS_TO_SCRAPE)

Search terms: ['leche', 'yogur', 'garbanzos', 'detergente']


In [2]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "aldi"

RAW_PATH = RAW_DIR / "aldi_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-20
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\aldi_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\aldi\2026-11-20_raw.csv


In [3]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies(driver):
    container = driver.execute_script("""
        function find(selector, root = document) {
            const el = root.querySelector(selector);
            if (el) return el;
            for (const node of root.querySelectorAll('*')) {
                if (node.shadowRoot) {
                    const found = find(selector, node.shadowRoot);
                    if (found) return found;
                }
            }
            return null;
        }
        return find('[data-testid="uc-container"]');
    """)

    if container is None:
        print("No apareció el popup de cookies.")
        return

    scroller = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-vertical-scroller"]');
    """, container)

    if scroller is not None:
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight;",
            scroller
        )

    time.sleep(0.5)

    accept_button = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-accept-all-button"]');
    """, container)

    if accept_button is None:
        raise RuntimeError("No se encontró el botón de aceptar cookies.")

    driver.execute_script("arguments[0].click();", accept_button)
    print("Cookies aceptadas.")



def select_peninsula_if_needed(driver):
    # Find the Peninsula region button
    buttons = driver.find_elements(
        By.CSS_SELECTOR,
        'button[data-region-code="pen"]'
    )

    if not buttons:
        print("Region selection not required.")
        return

    # Wait for the cookie banner to disappear
    try:
        WebDriverWait(driver, 10).until(
            lambda d: not any(
                element.is_displayed()
                for element in d.find_elements(
                    By.CSS_SELECTOR,
                    "#usercentrics-root"
                )
            )
        )
    except TimeoutException:
        print("Cookie overlay is still present.")

    # Try selecting the Peninsula region
    for button in buttons:
        if button.is_displayed() and button.is_enabled():

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                button
            )

            try:
                WebDriverWait(driver, 5).until(
                    lambda d: button.is_displayed()
                    and button.is_enabled()
                )

                button.click()
                print("Region selected: Peninsula")
                return

            except ElementClickInterceptedException:
                print(
                    "Region button is blocked by the cookie banner."
                )
                raise

    print("Peninsula region button not available.")



def go_to_home(driver):
    driver.get(HOME_URL)
    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(1)
    select_peninsula_if_needed(driver)

In [4]:
def find_search_box(driver):
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_input(_):
        for selector in selectors:
            for element in driver.find_elements(By.CSS_SELECTOR, selector):
                if element.is_displayed() and element.is_enabled():
                    return element
        return False

    return WebDriverWait(driver, WAIT).until(visible_input)


def search_product(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        lambda d: term.lower() in d.find_element(By.TAG_NAME, "body").text.lower()
    )

    WebDriverWait(driver, WAIT).until(
        lambda d: (
            len(d.find_elements(By.CSS_SELECTOR, ".product-tile")) > 0
            or "Productos (0)" in d.find_element(By.TAG_NAME, "body").text
        )
    )
    time.sleep(1)


def get_expected_product_count(driver):
    for tab in driver.find_elements(By.CSS_SELECTOR, '[data-testid="tab"]'):
        text = tab.text.strip()
        if text.startswith("Productos"):
            match = re.search(r"\((\d+)\)", text)
            if match:
                return int(match.group(1))
    return None


def get_loaded_product_count(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, ".product-tile"))



# --------------------------------------------------
# LOAD AVAILABLE PRODUCTS
# --------------------------------------------------

def load_all_products(driver, expected_count=None, max_clicks=40):

    loaded = get_loaded_product_count(driver)
    clicks = 0

    print(
        f"Initially loaded: {loaded} | "
        f"Website count: {expected_count}"
    )

    while clicks < max_clicks:

        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            '[data-testid="product-tile-grid-load-more-button"]'
        )

        buttons = [
            button
            for button in buttons
            if button.is_displayed() and button.is_enabled()
        ]

        # No more pagination is available.
        if not buttons:
            print("No more available pagination buttons.")
            break

        button = buttons[0]
        previous = loaded

        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            button
        )

        try:
            button.click()

        except Exception:
            driver.execute_script(
                "arguments[0].click();",
                button
            )

        # A failed pagination request is a technical error.
        # It will be retried after all other search terms.
        try:
            WebDriverWait(driver, WAIT).until(
                lambda d: (
                    get_loaded_product_count(d) > previous
                )
            )

        except TimeoutException as exc:
            raise RuntimeError(
                "Pagination did not load new products."
            ) from exc

        loaded = get_loaded_product_count(driver)
        clicks += 1

        print(
            f"Click {clicks}: {previous} -> {loaded} "
            f"| Website count: {expected_count}"
        )

    if clicks >= max_clicks:
        print(
            "WARNING: Maximum pagination clicks reached. "
            "Available products will be preserved."
        )

    if (
        expected_count is not None
        and loaded < expected_count
    ):
        print(
            f"WARNING: Website reports {expected_count} "
            f"products, but {loaded} cards were loaded."
        )

    return loaded, clicks


In [5]:
def extract_products(driver, searched_item):
    products = driver.execute_script("""
        function collectProducts(root = document, results = []) {
            const cards = root.querySelectorAll('.product-tile');

            for (const card of cards) {
                const brand = card.querySelector(
                    '.product-tile__content__upper__brand-name'
                );
                const name = card.querySelector(
                    '.product-tile__content__upper__product-name'
                );
                const price = card.querySelector(
                    '[data-testid$="-tag-current-price-amount"]'
                );
                const salesUnit = card.querySelector(
                    '[data-testid$="-tag-sales-unit"]'
                );
                const basePrice = card.querySelector(
                    '[data-testid$="-tag-base-price"]'
                );
                const link = card.querySelector('a.product-tile__action');

                results.push({
                    brand: brand ? brand.innerText.trim() : null,
                    name: name ? name.innerText.trim() : null,
                    price: price ? price.innerText.trim() : null,
                    quantity_raw: salesUnit ? salesUnit.innerText.trim() : null,
                    price_per_unit_raw: basePrice ? basePrice.innerText.trim() : null,
                    url: link ? link.href : null
                });
            }

            for (const el of root.querySelectorAll('*')) {
                if (el.shadowRoot) collectProducts(el.shadowRoot, results);
            }
            return results;
        }
        return collectProducts();
    """)

    df = pd.DataFrame(products)

    if df.empty:
        return df

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["product_id"] = df["url"].str.extract(
        r"-(\d+)\.html$", expand=False
    )
    df["supermarket"] = "ALDI"
    df["searched_item"] = searched_item
    df["scraped_at"] = datetime.now().isoformat(timespec="seconds")

    return df

In [6]:

# --------------------------------------------------
# ALDI COLLECTION WITH DEFERRED RETRIES
# --------------------------------------------------

# One initial pass + two additional retry passes.
MAX_COLLECTION_PASSES = MAX_RETRIES + 1

# Store only successful searches.
successful_results = {}

# Keep one final summary per search term.
summary_records = {}

driver = None


# --------------------------------------------------
# COLLECTION PASSES
# --------------------------------------------------

try:

    for pass_number in range(1, MAX_COLLECTION_PASSES + 1):

        pending_terms = [
            term
            for term in TERMS_TO_SCRAPE
            if term not in successful_results
        ]

        if not pending_terms:
            print("\nAll searches completed successfully.")
            break

        print("\n" + "=" * 70)
        print(
            f"COLLECTION PASS "
            f"{pass_number}/{MAX_COLLECTION_PASSES}"
        )
        print("Pending searches:", len(pending_terms))
        print("=" * 70)

        for i, term in enumerate(pending_terms, start=1):

            print(
                f"\n[{i}/{len(pending_terms)}] "
                f"ALDI - {term} "
                f"(attempt {pass_number})"
            )

            started = time.monotonic()

            status = "ERROR"
            error_message = None

            expected = None
            loaded = None
            extracted = 0
            clicks = None

            try:

                # --------------------------------------
                # START OR REUSE CHROME
                # --------------------------------------

                if driver is None:

                    print("Starting Chrome driver...")

                    driver = start_driver()

                    time.sleep(2)
                    accept_cookies(driver)
                    time.sleep(1)

                    select_peninsula_if_needed(driver)

                # --------------------------------------
                # SEARCH PRODUCTS
                # --------------------------------------

                go_to_home(driver)

                search_product(driver, term)

                # Website count is informational.
                expected = get_expected_product_count(driver)

                # --------------------------------------
                # LOAD PRODUCTS
                # --------------------------------------

                loaded, clicks = load_all_products(
                    driver,
                    expected
                )

                # --------------------------------------
                # EXTRACT PRODUCTS
                # --------------------------------------

                df_term = extract_products(
                    driver,
                    term
                )

                extracted = len(df_term)

                # Do not require an exact product count.
                # Only reject empty searches or technical errors.

                if df_term.empty:
                    raise RuntimeError(
                        "No products were extracted."
                    )

                # --------------------------------------
                # STORE SUCCESSFUL SEARCH
                # --------------------------------------

                successful_results[term] = df_term

                status = "OK"

                print(
                    f"SUCCESS: {extracted} products extracted."
                )

                # Differences are warnings, not failures.
                if (
                    expected is not None
                    and extracted != expected
                ):
                    print(
                        f"WARNING: Website count={expected}, "
                        f"extracted={extracted}."
                    )

                if loaded != extracted:
                    print(
                        f"WARNING: Loaded cards={loaded}, "
                        f"extracted={extracted}."
                    )

            except Exception as exc:

                error_message = (
                    f"{type(exc).__name__}: {exc}"
                )

                print(
                    f"FAILED: {term} - {error_message}"
                )

                print(
                    "Continuing with the next search. "
                    "This term will be retried later."
                )

                # Discard products from failed attempts.
                extracted = 0

                # Restart the browser after an error.
                if driver is not None:

                    try:
                        driver.quit()
                    except Exception:
                        pass

                    driver = None

            elapsed = round(
                time.monotonic() - started,
                2
            )

            previous_seconds = summary_records.get(
                term, {}
            ).get("seconds", 0)

            summary_records[term] = {
                "searched_item": term,
                "expected_products": expected,
                "loaded_products": loaded,
                "extracted_products": extracted,
                "load_more_clicks": clicks,
                "attempts": pass_number,
                "status": status,
                "seconds": round(
                    previous_seconds + elapsed,
                    2
                ),
                "error": error_message,
            }

        # ----------------------------------------------
        # PASS SUMMARY
        # ----------------------------------------------

        remaining_terms = [
            term
            for term in TERMS_TO_SCRAPE
            if term not in successful_results
        ]

        print("\nPASS SUMMARY")
        print(
            "Successful searches:",
            len(successful_results)
        )
        print(
            "Remaining failures:",
            len(remaining_terms)
        )

        if (
            remaining_terms
            and pass_number < MAX_COLLECTION_PASSES
        ):

            print("\nSearches to retry:")

            for term in remaining_terms:
                print(f"  - {term}")

            time.sleep(10)


finally:

    # Always close the browser after collection.
    if driver is not None:

        try:
            driver.quit()
        except Exception:
            pass

    driver = None


# --------------------------------------------------
# BUILD FINAL RESULTS
# --------------------------------------------------

# Include each successful term exactly once.
all_results = [
    successful_results[term]
    for term in TERMS_TO_SCRAPE
    if term in successful_results
]

# Exactly one summary row per search term.
summary = [
    summary_records[term]
    for term in TERMS_TO_SCRAPE
]

df_summary = pd.DataFrame(summary)


# --------------------------------------------------
# FINAL COLLECTION REPORT
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL ALDI COLLECTION SUMMARY")
print("=" * 70)

display(df_summary)

successful_count = int(
    df_summary["status"].eq("OK").sum()
)

recovered_searches = df_summary[
    df_summary["status"].eq("OK")
    & df_summary["attempts"].gt(1)
]

failed_searches = df_summary[
    df_summary["status"].ne("OK")
]

print("\nExpected searches:", len(TERMS_TO_SCRAPE))
print("Successful searches:", successful_count)
print("Recovered after retry:", len(recovered_searches))
print("Final failures:", len(failed_searches))

print(
    "Total extracted rows:",
    sum(len(frame) for frame in all_results)
)

# --------------------------------------------------
# RECOVERED SEARCHES
# --------------------------------------------------

if not recovered_searches.empty:

    print("\nRECOVERED SEARCHES:")

    display(
        recovered_searches[
            [
                "searched_item",
                "attempts",
                "extracted_products",
                "seconds"
            ]
        ]
    )


# --------------------------------------------------
# FINAL FAILURES
# --------------------------------------------------

if not failed_searches.empty:

    print("\nWARNING: Some searches remain incomplete.")

    display(
        failed_searches[
            [
                "searched_item",
                "attempts",
                "status",
                "error"
            ]
        ]
    )

    print(
        "Incomplete collections will not be published."
    )

else:

    print("\nAll searches completed successfully.")
    print("RAW data is ready for validation and saving.")



COLLECTION PASS 1/3
Pending searches: 4

[1/4] ALDI - leche (attempt 1)
Starting Chrome driver...
No apareció el popup de cookies.
Region button is blocked by the cookie banner.
FAILED: leche - ElementClickInterceptedException: Message: element click intercepted: Element <button data-region-code="pen" type="button">...</button> is not clickable at point (361, 305). Other element would receive the click: <div id="usercentrics-root" data-created-at="1795172621983" style=""></div>
  (Session info: chrome=158.0.8127.36); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff6be180f25]
	(No symbol) [0x7ff6be081dd0]
	(No symbol) [0x7ff6bdf03433]
	(No symbol) [0x7ff6be74bf22]
	(No symbol) [0x7ff6be74a53a]
	(No symbol) [0x7ff6be74842f]
	(No symbol) [0x7ff6be747bc0]
	(No symbol) [0x7ff6be73c842]
	(No symbol) [0x7ff

,searched_item,expected_products,loaded_products,extracted_products,load_more_clicks,attempts,status,seconds,error
0,leche,133,133,133,14,2,OK,119.64,None
1,yogur,117,117,117,12,1,OK,51.12,None
2,garbanzos,7,7,7,0,1,OK,5.91,None
3,detergente,12,12,12,1,1,OK,8.08,None



Expected searches: 4
Successful searches: 4
Recovered after retry: 1
Final failures: 0
Total extracted rows: 269

RECOVERED SEARCHES:


,searched_item,attempts,extracted_products,seconds
0,leche,2,133,119.64



All searches completed successfully.
RAW data is ready for validation and saving.


## RAW — Combine All Search Results into a Single Dataset

In [7]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if not all_results:
    raise RuntimeError("No products were collected.")

df_aldi_raw = pd.concat(
    all_results,
    ignore_index=True
)

print("Raw rows:", len(df_aldi_raw))
print("Unique product IDs:", df_aldi_raw["product_id"].nunique())
print("Raw data ready for processing.")


Raw rows: 269
Unique product IDs: 260
Raw data ready for processing.


In [8]:

# --------------------------------------------------
# BROWSER CLEANUP
# --------------------------------------------------

# Chrome is already closed in the collection cell.
print("Chrome session already closed.")


Chrome session already closed.


In [10]:

# --------------------------------------------------
# VALIDATE AND SAVE CURRENT AND HISTORICAL RAW DATA
# --------------------------------------------------

import os
import tempfile


if TEST_MODE:

    print("TEST MODE: no files will be saved.")

    print("\nTEST COLLECTION SUMMARY")
    print("Searches attempted:", len(df_summary))

    display(df_summary)

    failed_searches = df_summary[
        df_summary["status"] != "OK"
    ]

    if failed_searches.empty:
        print("\nTEST PASSED: all test searches succeeded.")
    else:
        print("\nTEST WARNING: some searches failed.")
        display(failed_searches)


else:

    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_aldi_raw.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded. No files saved."
        )

    required_columns = [
        "brand",
        "name",
        "price",
        "quantity_raw",
        "price_per_unit_raw",
        "url",
        "product_id",
        "supermarket",
        "searched_item",
        "scraped_at",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df_aldi_raw.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    # --------------------------------------------------
    # CHECK COLLECTION COMPLETENESS
    # --------------------------------------------------

    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and set(df_summary["searched_item"]) == set(SEARCH_TERMS)
        and df_summary["searched_item"].is_unique
    )

    successful_searches = (
        df_summary["status"].eq("OK").all()
    )

    collection_complete = bool(
        all_terms_attempted and successful_searches
    )

    # --------------------------------------------------
    # PREVENT INCOMPLETE COLLECTIONS FROM BEING SAVED
    # --------------------------------------------------

    if not collection_complete:

        print("\nERROR: Incomplete ALDI collection.")

        failed_searches = df_summary[
            df_summary["status"] != "OK"
        ]

        if not failed_searches.empty:

            display(
                failed_searches[
                    [
                        "searched_item",
                        "status",
                        "attempts",
                        "error"
                    ]
                ]
            )

        raise RuntimeError(
            "ALDI collection is incomplete after "
            "all retry attempts. No RAW files were saved."
        )

    # --------------------------------------------------
    # VALIDATE RAW PRODUCT DATA
    # --------------------------------------------------

    numeric_prices = pd.to_numeric(
        df_aldi_raw["price"],
        errors="coerce"
    )

    valid_prices = (
        numeric_prices.notna()
        & numeric_prices.gt(0)
    )

    valid_names = (
        df_aldi_raw["name"].notna()
        & df_aldi_raw["name"].astype(str).str.strip().ne("")
    )

    valid_rows = valid_prices & valid_names

    if not valid_rows.any():
        raise RuntimeError(
            "No products with valid names and prices found. "
            "No files saved."
        )

    invalid_rows = int((~valid_rows).sum())

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} RAW rows have "
            "missing names or invalid prices."
        )
        print(
            "These rows will be preserved for "
            "central pipeline validation."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists "
            f"for {RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE OUTPUT DIRECTORIES
    # --------------------------------------------------

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------
    # PREPARE TEMPORARY RAW CSV FILES
    # --------------------------------------------------

    def prepare_raw_temp(destination):

        fd, temp_name = tempfile.mkstemp(
            prefix=".aldi_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:

            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:

                df_aldi_raw.to_csv(
                    file,
                    index=False
                )

            return Path(temp_name)

        except Exception:

            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:

        history_temp = prepare_raw_temp(
            RAW_HISTORY_PATH
        )

        current_temp = prepare_raw_temp(
            RAW_PATH
        )

        # Save historical RAW without overwriting it.
        with history_temp.open("rb") as source:

            with RAW_HISTORY_PATH.open("xb") as target:

                while True:

                    chunk = source.read(1024 * 1024)

                    if not chunk:
                        break

                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Update current RAW after saving history.
        os.replace(
            current_temp,
            RAW_PATH
        )

        print("Saved current RAW:", RAW_PATH)

    finally:

        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # FINAL COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Collection date:", RUN_DATE)
    print("Status: COMPLETE")
    print("RAW rows:", len(df_aldi_raw))

    print(
        "Distinct product IDs:",
        df_aldi_raw["product_id"].nunique()
    )

    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))
    print("Rows requiring validation:", invalid_rows)

    print("\nSaved outputs:")
    print(RAW_PATH)
    print(RAW_HISTORY_PATH)


TEST MODE: no files will be saved.

TEST COLLECTION SUMMARY
Searches attempted: 4


,searched_item,expected_products,loaded_products,extracted_products,load_more_clicks,attempts,status,seconds,error
0,leche,133,133,133,14,2,OK,119.64,None
1,yogur,117,117,117,12,1,OK,51.12,None
2,garbanzos,7,7,7,0,1,OK,5.91,None
3,detergente,12,12,12,1,1,OK,8.08,None



TEST PASSED: all test searches succeeded.
